In [ ]:
import tensorflow as tf
from keras.src.layers import Dropout
from tensorflow.keras import layers
import tensorboard as tb
import pandas as pd
import numpy as np
import datetime as dt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

print("TensorFlow version: ", tf.__version__)

pjm_hourly_est = pd.read_csv('pjm_hourly_est.csv')

print(pjm_hourly_est.head())


In [ ]:
#create test dataset, convert NA values to 0, and sum across rows
test_data = pjm_hourly_est.copy()
numeric_cols = test_data.select_dtypes(include=[np.number]).columns

test_data[numeric_cols] = test_data[numeric_cols].fillna(0)
test_data['total_usage'] = test_data[numeric_cols].sum(axis=1, skipna=True)

test_data = test_data[['Datetime', 'total_usage']]

print(test_data.head())

In [ ]:
#convert string Datetime to datetime object
print(test_data.dtypes)

format_string = "%Y-%m-%d %H:%M:%S"

test_data['Datetime'] = pd.to_datetime(test_data['Datetime'], format='%Y-%m-%d %H:%M:%S')

print(test_data.dtypes)

In [ ]:
#create modified dataset with offset sinusoidal waves for date and time

input_scaler = MinMaxScaler(feature_range=(-1, 1))
output_scaler = MinMaxScaler(feature_range=(0,1))

data_mod = test_data.copy()
data_mod['year'] = data_mod['Datetime'].dt.year
data_mod['day_of_year'] = data_mod['Datetime'].dt.dayofyear
data_mod['time_of_day'] = data_mod['Datetime'].dt.hour

data_mod['year_scaled'] = input_scaler.fit_transform(data_mod[['year']])

data_mod['day_SIN'] = np.sin((2 * np.pi * data_mod['day_of_year']) / 365)
data_mod['day_COS'] = np.cos((2 * np.pi * data_mod['day_of_year']) / 365)
data_mod['time_SIN'] = np.sin((2 * np.pi * data_mod['time_of_day']) / 24)
data_mod['time_COS'] = np.cos((2 * np.pi * data_mod['time_of_day']) / 24)

data_mod['total_usage_scaled'] = output_scaler.fit_transform(data_mod[['total_usage']])

data_mod = data_mod.sort_values('Datetime')

#copy modified data to df
df = data_mod.copy()

print(df.head())

In [ ]:
feature_cols = ['year_scaled', 'day_SIN', 'day_COS', 'time_SIN', 'time_COS']
target_col = 'total_usage_scaled'

#first 80% of data is training, last 20% is testing
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

print(f"Train range: {train_df['Datetime'].min()} to {train_df['Datetime'].max()}")
print(f"Test range: {test_df['Datetime'].min()} to {test_df['Datetime'].max()}")

In [ ]:
#function to create sequences for the LSTM to interpret
def create_sequences(data, target, window_size):
    x, y = [], []
    for i in range(len(data) - window_size):
        x.append(data[i:(i + window_size)])
        y.append(target[i + window_size])
    return np.array(x), np.array(y)

In [ ]:
WINDOW_SIZE = 24;

X_train, y_train = create_sequences(
    train_df[feature_cols].values,
    train_df[target_col].values,
    WINDOW_SIZE
)

X_test, y_test = create_sequences(
    test_df[feature_cols].values,
    test_df[target_col].values,
    WINDOW_SIZE
)

#check to make sure X_train shape is ([num_samples], 24, 5)
print("X_train shape:", X_train.shape)

In [ ]:
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input, Dropout

model = Sequential([
    Input(shape = (24,5)),

    LSTM(BATCH_SIZE, activation='tanh'),

    Dropout(0.2),

    Dense(1),
])

model.compile(
    optimizer='adam',
    loss='mse',      # Mean Squared Error
    metrics=['mae']  # Mean Absolute Error
)

In [ ]:
EPOCHS = 20

history = model.fit(
    train_dataset,
    epochs=EPOCHS,
    validation_data=test_dataset,
    verbose=1
)

In [ ]:
predictions_scaled = model.predict(test_dataset)

predictions_mw = output_scaler.inverse_transform(predictions_scaled)

y_test_mw = output_scaler.inverse_transform(y_test.reshape(-1, 1))

In [ ]:
preds_pd = pd.DataFrame(predictions_mw)
test_pd = pd.DataFrame(y_test_mw)
pred_vs_test = pd.concat([preds_pd, test_pd], axis = 1)
pred_vs_test.columns = ['Predicted', 'Actual']
print(current_columns)
print(pred_vs_test.head())

print(f"First Prediction: {predictions_mw[0][0]:.2f} MW")
print(f"Actual Usage: {y_test_mw[0][0]:.2f} MW")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score

x = pred_vs_test['Predicted']
y = pred_vs_test['Actual']

#COPIED OVER: get r^2 for prediction and actual
z = np.polyfit(x, y, 1)
p = np.poly1d(z)
y_pred = p(x)
r_squared = r2_score(y, y_pred)
print(r_squared)


plt.scatter(x,y, alpha=0.01)
plt.plot(x,p(x),"r--")

plt.show()